# COMP34812 NLI — Input Embedding Module

**Author:** Kaan Berk Temizel  
**Track:** Natural Language Inference (NLI)  
**Solution:** Category B — Deep Learning (non-transformer)

## Overview
This notebook implements the input embedding pipeline for a ResESIM-based NLI model.
Each token is represented by a 1476-dimensional vector formed by concatenating:

| Component | Dim | Type | Reference |
|---|---|---|---|
| GloVe | 300 | Static, frozen | Pennington et al., 2014 |
| ELMo | 1024 | Contextual, frozen | Peters et al., 2018 |
| CharCNN | 100 | Learned | Kim et al., 2016 |
| POS embeddings | 50 | Learned | — |
| Negation flags | 2 | Rule-based | spaCy dep parse |
| **Total** | **1476** | | |

## Notebook Structure
1. Environment Setup
2. Data Loading
3. Tokenisation & Vocabulary
4. GloVe Embeddings
5. Character CNN
6. POS Embeddings
7. Negation Flags
8. ELMo Pre-computation
9. Input Embedding Module
10. Dataset & DataLoaders
11. Save to Drive

## 1. Environment Setup

ELMo (Peters et al., 2018) requires AllenNLP which is incompatible with Python 3.11.
We create a Python 3.10 virtual environment solely for ELMo pre-computation.
All other components run in the main kernel.

In [ ]:
# ── Main kernel dependencies ──────────────────────────────────────────────────
!pip install -q spacy psutil
!python -m spacy download en_core_web_sm -q

# ── GloVe weights (Pennington et al., 2014) ───────────────────────────────────
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

# ── ELMo weights (Peters et al., 2018) — original AllenNLP release ───────────
import os
os.makedirs('/content/elmo_model', exist_ok=True)
!wget -q -O /content/elmo_model/options.json \
  https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_options.json
!wget -q -O /content/elmo_model/weights.hdf5 \
  https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5

# ── Python 3.10 venv for AllenNLP ELMo pre-computation ───────────────────────
!pip install -q virtualenv
!virtualenv -p /usr/bin/python3.10 /content/myenv -q
!source /content/myenv/bin/activate && \
  /content/myenv/bin/pip install -q allennlp allennlp-models && \
  /content/myenv/bin/pip install -q 'numpy<2.0' --force-reinstall && \
  /content/myenv/bin/pip install -q torch==1.13.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117 && \
  /content/myenv/bin/pip install -q \
    https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.3.0/en_core_web_sm-3.3.0-py3-none-any.whl

print('Environment setup complete.')

In [ ]:
# ── Main kernel imports ───────────────────────────────────────────────────────
import re
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import spacy
nlp = spacy.load('en_core_web_sm')

# ── Constants ─────────────────────────────────────────────────────────────────
PAD_TOKEN    = '<PAD>'
UNK_TOKEN    = '<UNK>'
PAD_CHAR     = '<PAD>'
UNK_CHAR     = '<UNK>'
PAD_POS      = '<PAD>'
UNK_POS      = '<UNK>'
MAX_LEN      = 64
MAX_WORD_LEN = 20
MIN_FREQ     = 2
EMBED_DIM    = 300
CHAR_EMBED_DIM = 30
CHAR_OUT_DIM   = 100
POS_EMBED_DIM  = 50
NUM_CLASSES    = 2

NEGATION_WORDS = {
    'not', 'no', 'never', 'neither', 'nor', 'nobody', 'nothing',
    'nowhere', 'hardly', 'barely', 'scarcely', "n't", 'cannot'
}

TRAIN_PATH = Path('/content/train.csv')
DEV_PATH   = Path('/content/dev.csv')
TRIAL_PATH = Path('/content/NLI_trial.csv')
GLOVE_PATH = Path('glove.6B.300d.txt')

print('Imports and constants OK')

## 2. Data Loading

In [ ]:
def load_nli_csv(path):
    """Load NLI CSV with columns: premise, hypothesis, label."""
    df = pd.read_csv(Path(path))
    assert {'premise', 'hypothesis', 'label'}.issubset(df.columns), \
        f'Missing expected columns in {Path(path).name}'
    before = len(df)
    df = df.dropna(subset=['premise', 'hypothesis', 'label']).reset_index(drop=True)
    if before != len(df):
        print(f'  Dropped {before - len(df)} rows with NaN')
    df['label'] = df['label'].astype(int)
    print(f'{Path(path).name}: {len(df)} pairs')
    print(df['label'].value_counts().sort_index())
    print()
    return df

train_df = load_nli_csv(TRAIN_PATH)
dev_df   = load_nli_csv(DEV_PATH)
trial_df = load_nli_csv(TRIAL_PATH)

## 3. Tokenisation & Vocabulary

In [ ]:
def tokenise(text):
    """
    Lowercase and tokenise into word strings.
    Handles contractions (don't, I'm) and strips punctuation.
    """
    return re.findall(r"[a-z]+(?:'[a-z]+)*", text.lower().strip())

# tokenise all splits
train_prem_tokens = [tokenise(s) for s in train_df['premise']]
train_hyp_tokens  = [tokenise(s) for s in train_df['hypothesis']]
dev_prem_tokens   = [tokenise(s) for s in dev_df['premise']]
dev_hyp_tokens    = [tokenise(s) for s in dev_df['hypothesis']]

print('Sample premise:   ', train_df['premise'][0])
print('Tokenised:        ', train_prem_tokens[0])

In [ ]:
class Vocabulary:
    """
    Word-to-index mapping built from training data only.
    Index 0 = <PAD>, Index 1 = <UNK>.
    Words appearing fewer than min_freq times map to <UNK>.
    """
    def __init__(self, min_freq=MIN_FREQ):
        self.min_freq  = min_freq
        self.word2idx  = {PAD_TOKEN: 0, UNK_TOKEN: 1}
        self.idx2word  = {0: PAD_TOKEN, 1: UNK_TOKEN}
        self.word_freq = Counter()

    def build(self, token_lists):
        for tokens in token_lists:
            self.word_freq.update(tokens)
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
        print(f'Vocabulary size: {len(self.word2idx)} tokens (min_freq={self.min_freq})')

    def encode(self, tokens):
        unk = self.word2idx[UNK_TOKEN]
        return [self.word2idx.get(t, unk) for t in tokens]

    def __len__(self):
        return len(self.word2idx)

# build from training set only — no dev/test leakage
vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(train_prem_tokens + train_hyp_tokens)

## 4. GloVe Embeddings

GloVe (Pennington et al., 2014) provides static 300-dimensional word vectors.
The embedding matrix is frozen during training — weights are not updated.
Words not found in GloVe are randomly initialised from U(-0.1, 0.1).
The <UNK> token is set to the mean of all GloVe vectors.

In [ ]:
def load_glove(glove_path, vocab, embed_dim=EMBED_DIM):
    """Build embedding matrix [vocab_size, embed_dim] from GloVe file."""
    vocab_size = len(vocab)
    matrix = np.random.uniform(-0.1, 0.1, (vocab_size, embed_dim))
    matrix[0] = np.zeros(embed_dim)  # PAD = zeros

    glove_vectors = {}
    print('Reading GloVe file...')
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            glove_vectors[parts[0]] = np.array(parts[1:], dtype=np.float32)
    print(f'GloVe loaded: {len(glove_vectors)} vectors')

    found = 0
    for word, idx in vocab.word2idx.items():
        if word in glove_vectors:
            matrix[idx] = glove_vectors[word]
            found += 1
    matrix[1] = np.mean(list(glove_vectors.values()), axis=0)  # UNK = mean

    print(f'Words found in GloVe:     {found}')
    print(f'Words not found (random): {vocab_size - found}')
    print(f'Embedding matrix shape:   {matrix.shape}')
    return matrix

def build_glove_layer(glove_matrix):
    """Wrap GloVe matrix in a frozen PyTorch Embedding layer."""
    vocab_size, embed_dim = glove_matrix.shape
    layer = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
    layer.weight = nn.Parameter(
        torch.tensor(glove_matrix, dtype=torch.float32),
        requires_grad=False
    )
    print(f'GloVe layer: {vocab_size} x {embed_dim}, frozen=True')
    return layer

glove_matrix = load_glove(GLOVE_PATH, vocab)
glove_layer  = build_glove_layer(glove_matrix)

## 5. Character CNN

Character-level CNN (Kim et al., 2016) produces 100-dimensional vectors per word.
Uses three parallel 1D convolutions with kernel sizes 2, 3, 4 followed by max pooling.
This handles out-of-vocabulary words that GloVe misses (contractions, rare words).

In [ ]:
def build_char_vocab(token_lists):
    """Build character vocabulary from training tokens. 0=PAD, 1=UNK."""
    char2idx = {PAD_CHAR: 0, UNK_CHAR: 1}
    for tokens in token_lists:
        for token in tokens:
            for char in token:
                if char not in char2idx:
                    char2idx[char] = len(char2idx)
    print(f'Character vocabulary size: {len(char2idx)}')
    return char2idx

char2idx = build_char_vocab(train_prem_tokens + train_hyp_tokens)

def words_to_char_tensor(token_list, char2idx,
                          max_word_len=MAX_WORD_LEN, max_seq_len=MAX_LEN):
    """Convert token list to character index tensor [max_seq_len, max_word_len]."""
    pad = char2idx[PAD_CHAR]
    unk = char2idx[UNK_CHAR]
    result = []
    for i in range(max_seq_len):
        if i < len(token_list):
            word = token_list[i][:max_word_len]
            ids  = [char2idx.get(c, unk) for c in word]
            ids  = ids + [pad] * (max_word_len - len(ids))
        else:
            ids = [pad] * max_word_len
        result.append(ids)
    return torch.tensor(result, dtype=torch.long)

class CharCNN(nn.Module):
    """
    Character-level CNN producing 100d vector per word.
    Three parallel convolutions (kernels 2,3,4) + max pooling + concat.
    Reference: Kim et al. (2016), Character-Aware Neural Language Models.
    """
    def __init__(self, char_vocab_size,
                 char_embed_dim=CHAR_EMBED_DIM, out_dim=CHAR_OUT_DIM):
        super().__init__()
        self.char_embedding = nn.Embedding(
            char_vocab_size, char_embed_dim, padding_idx=0)
        self.num_filters = out_dim // 3
        self.conv2 = nn.Conv1d(char_embed_dim, self.num_filters,     kernel_size=2)
        self.conv3 = nn.Conv1d(char_embed_dim, self.num_filters,     kernel_size=3)
        self.conv4 = nn.Conv1d(char_embed_dim, self.num_filters + 1, kernel_size=4)
        self.dropout = nn.Dropout(0.3)

    def forward(self, char_ids):
        # char_ids: [batch, seq_len, max_word_len]
        batch, seq_len, max_word_len = char_ids.shape
        char_ids = char_ids.view(batch * seq_len, max_word_len)
        x = self.char_embedding(char_ids).transpose(1, 2)
        x2 = F.max_pool1d(F.relu(self.conv2(x)), self.conv2(x).shape[-1]).squeeze(-1)
        x3 = F.max_pool1d(F.relu(self.conv3(x)), self.conv3(x).shape[-1]).squeeze(-1)
        x4 = F.max_pool1d(F.relu(self.conv4(x)), self.conv4(x).shape[-1]).squeeze(-1)
        out = self.dropout(torch.cat([x2, x3, x4], dim=-1))
        return out.view(batch, seq_len, -1)  # [batch, seq_len, 100]

char_cnn = CharCNN(char_vocab_size=len(char2idx))
print(f'CharCNN output dim: {CHAR_OUT_DIM}')

## 6. POS Embeddings

Learned 50-dimensional embeddings for Universal POS tags (spaCy).
Unlike GloVe, these are updated during training.
Provides explicit syntactic signal complementary to ELMo's implicit encoding.

In [ ]:
def build_pos_vocab(token_lists):
    """Collect POS tags from training data via spaCy. 0=PAD, 1=UNK."""
    pos2idx = {PAD_POS: 0, UNK_POS: 1}
    for tokens in token_lists:
        doc = nlp(' '.join(tokens))
        for token in doc:
            if token.pos_ not in pos2idx:
                pos2idx[token.pos_] = len(pos2idx)
    print(f'POS vocabulary: {len(pos2idx)} tags')
    print(f'Tags: {list(pos2idx.keys())}')
    return pos2idx

def get_pos_ids(token_list, pos2idx, max_len=MAX_LEN):
    """Return POS tag indices for a token list, padded to max_len."""
    unk = pos2idx[UNK_POS]
    pad = pos2idx[PAD_POS]
    doc = nlp(' '.join(token_list))
    ids = [pos2idx.get(t.pos_, unk) for t in doc]
    ids = ids[:max_len] + [pad] * (max_len - len(ids))
    return ids

class POSEmbedding(nn.Module):
    """Learned POS tag embedding layer. Updated during training."""
    def __init__(self, pos_vocab_size, pos_embed_dim=POS_EMBED_DIM):
        super().__init__()
        self.embedding = nn.Embedding(
            pos_vocab_size, pos_embed_dim, padding_idx=0)

    def forward(self, pos_ids):
        return self.embedding(pos_ids)  # [batch, seq_len, 50]

pos2idx       = build_pos_vocab(train_prem_tokens + train_hyp_tokens)
pos_embedding = POSEmbedding(pos_vocab_size=len(pos2idx))

## 7. Negation Flags

Rule-based 2-dimensional binary feature per token:
- dim 0: is this token a negation word?
- dim 1: is this token in the scope of a negation? (head verb of neg dependency)

Motivation: NLI models frequently fail on negation. GloVe has no context awareness
for negation scope. ELMo captures it implicitly but this explicit signal helps the
negation gate in the attention module.

In [ ]:
def get_negation_flags(token_list, max_len=MAX_LEN):
    """
    Returns [max_len, 2] float32 tensor.
    dim 0: is_neg  — token is a negation word or has neg dependency
    dim 1: in_scope — token is the head verb of a negation
    """
    doc = nlp(' '.join(token_list[:max_len]))
    negated_heads = {t.head.i for t in doc if t.dep_ == 'neg'}
    flags = []
    for t in doc:
        is_neg   = 1.0 if t.text in NEGATION_WORDS or t.dep_ == 'neg' else 0.0
        in_scope = 1.0 if t.i in negated_heads else 0.0
        flags.append([is_neg, in_scope])
    flags = flags[:max_len] + [[0.0, 0.0]] * (max_len - len(flags))
    return torch.tensor(flags, dtype=torch.float32)

## 8. ELMo Pre-computation

ELMo (Peters et al., 2018) produces contextual 1024-dimensional embeddings.
Unlike GloVe, ELMo uses a 2-layer BiLSTM over the full sentence, producing
different vectors for the same word in different contexts.

**Implementation note:** AllenNLP (the original ELMo library) requires Python ≤ 3.10
and PyTorch < 1.13. We use a Python 3.10 venv to run pre-computation once,
save vectors to disk, then load them in the main kernel for training.
This is consistent with standard practice (Peters et al. recommend pre-computing
ELMo representations for downstream tasks).

In [ ]:
# ── Write ELMo pre-computation script ────────────────────────────────────────
elmo_script = '''
import numpy as np
import pandas as pd
import re
import torch
from allennlp.modules.elmo import Elmo, batch_to_ids

TRAIN_PATH   = "/content/train.csv"
DEV_PATH     = "/content/dev.csv"
ELMO_OPTIONS = "/content/elmo_model/options.json"
ELMO_WEIGHTS = "/content/elmo_model/weights.hdf5"
MAX_LEN      = 64
BATCH_SIZE   = 64

print("Loading ELMo (Peters et al., 2018) via AllenNLP...")
elmo   = Elmo(ELMO_OPTIONS, ELMO_WEIGHTS, num_output_representations=1, dropout=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
elmo   = elmo.to(device).eval()
print(f"ELMo loaded on {device}")

def tokenise(text):
    return re.findall(r"[a-z]+(?:[apostrophe][a-z]+)*", text.lower().strip())

def load_csv(path):
    df = pd.read_csv(path)
    return df["premise"].tolist(), df["hypothesis"].tolist()

def compute_elmo(sentences, name):
    print(f"Computing ELMo for {len(sentences)} {name} sentences...")
    token_lists = [tokenise(s) for s in sentences]
    token_lists = [t[:MAX_LEN] if len(t) > 0 else ["unk"] for t in token_lists]
    all_embeddings = np.zeros((len(token_lists), MAX_LEN, 1024), dtype=np.float32)
    for i in range(0, len(token_lists), BATCH_SIZE):
        batch    = token_lists[i:i+BATCH_SIZE]
        char_ids = batch_to_ids(batch).to(device)
        with torch.no_grad():
            output = elmo(char_ids)
        emb = output["elmo_representations"][0].cpu().numpy()
        for j in range(len(batch)):
            seq_len = min(len(batch[j]), MAX_LEN)
            all_embeddings[i+j, :seq_len, :] = emb[j, :seq_len, :]
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {min(i+BATCH_SIZE, len(token_lists))}/{len(token_lists)}")
    print(f"  Done. Shape: {all_embeddings.shape}")
    return all_embeddings

train_prem, train_hyp = load_csv(TRAIN_PATH)
np.save("/content/elmo_train_prem.npy", compute_elmo(train_prem, "train premises"))
np.save("/content/elmo_train_hyp.npy",  compute_elmo(train_hyp,  "train hypotheses"))
print("Train saved.")

dev_prem, dev_hyp = load_csv(DEV_PATH)
np.save("/content/elmo_dev_prem.npy", compute_elmo(dev_prem, "dev premises"))
np.save("/content/elmo_dev_hyp.npy",  compute_elmo(dev_hyp,  "dev hypotheses"))
print("Dev saved.")
print("ALL DONE.")
'''

elmo_script = elmo_script.replace('[apostrophe]', "[\'']")
with open('/content/elmo_precompute.py', 'w') as f:
    f.write(elmo_script)
print('elmo_precompute.py written.')

In [ ]:
# ── Run ELMo pre-computation (~15 mins on A100) ───────────────────────────────
# Skip this cell if elmo_arrays.npz already exists on Drive
!/content/myenv/bin/python /content/elmo_precompute.py

In [ ]:
# ── Load pre-computed ELMo arrays ────────────────────────────────────────────
# Option A: load from local /content (if just pre-computed above)
elmo_train_prem = np.load('/content/elmo_train_prem.npy')
elmo_train_hyp  = np.load('/content/elmo_train_hyp.npy')
elmo_dev_prem   = np.load('/content/elmo_dev_prem.npy')
elmo_dev_hyp    = np.load('/content/elmo_dev_hyp.npy')

# Option B: load from Drive (if restoring after session reset)
# from google.colab import drive
# drive.mount('/content/drive')
# elmo = np.load('/content/drive/MyDrive/COMP34812_NLI/elmo_arrays.npz')
# elmo_train_prem = elmo['train_prem']
# elmo_train_hyp  = elmo['train_hyp']
# elmo_dev_prem   = elmo['dev_prem']
# elmo_dev_hyp    = elmo['dev_hyp']

print(f'elmo_train_prem: {elmo_train_prem.shape}')
print(f'elmo_train_hyp:  {elmo_train_hyp.shape}')
print(f'elmo_dev_prem:   {elmo_dev_prem.shape}')
print(f'elmo_dev_hyp:    {elmo_dev_hyp.shape}')

## 9. Input Embedding Module

Combines all five components into a single [batch, seq_len, 1476] tensor.
This is the interface between the embedding pipeline and ResESIM.

In [ ]:
class InputEmbeddingModule(nn.Module):
    """
    Combines GloVe + ELMo + CharCNN + POS + Negation flags.
    Output: [batch, seq_len, 1476]

    Component breakdown:
      GloVe  [300d] — Pennington et al. (2014), frozen
      ELMo  [1024d] — Peters et al. (2018), pre-computed frozen
      CharCNN[100d] — Kim et al. (2016), learned
      POS    [ 50d] — learned
      Negation[ 2d] — rule-based spaCy dependency parse
    """
    def __init__(self, glove_layer, char_cnn, pos_embedding, dropout=0.5):
        super().__init__()
        self.glove_layer   = glove_layer
        self.char_cnn      = char_cnn
        self.pos_embedding = pos_embedding
        self.dropout       = nn.Dropout(dropout)
        self.output_dim    = 300 + 1024 + 100 + 50 + 2  # 1476

    def forward(self, token_ids, char_ids, pos_ids, neg_flags, elmo_vecs):
        glove_out = self.glove_layer(token_ids)    # [batch, seq_len, 300]
        char_out  = self.char_cnn(char_ids)        # [batch, seq_len, 100]
        pos_out   = self.pos_embedding(pos_ids)    # [batch, seq_len,  50]
        elmo_out  = elmo_vecs.float()              # [batch, seq_len,1024]
        out = torch.cat([glove_out, elmo_out, char_out, pos_out, neg_flags], dim=-1)
        return self.dropout(out)                   # [batch, seq_len,1476]

input_module = InputEmbeddingModule(
    glove_layer   = glove_layer,
    char_cnn      = char_cnn,
    pos_embedding = pos_embedding,
    dropout       = 0.5
)

print(f'InputEmbeddingModule output dim: {input_module.output_dim}')
print(f'Trainable parameters: {sum(p.numel() for p in input_module.parameters() if p.requires_grad):,}')
print(f'Frozen parameters:    {sum(p.numel() for p in input_module.parameters() if not p.requires_grad):,}')

## 10. Dataset & DataLoaders

NLIDataset returns all tensors needed by OracleNet in one batch dict.
No on-the-fly ELMo computation — vectors are loaded from pre-computed arrays.

In [ ]:
class NLIDataset(Dataset):
    """
    PyTorch Dataset for NLI pairs.
    Each item returns a dict with all tensors ready for InputEmbeddingModule + OracleNet.

    Keys:
      premise_ids      [MAX_LEN]         int64  — GloVe token indices
      hypothesis_ids   [MAX_LEN]         int64  — GloVe token indices
      premise_char     [MAX_LEN, 20]     int64  — CharCNN input
      hypothesis_char  [MAX_LEN, 20]     int64  — CharCNN input
      premise_pos      [MAX_LEN]         int64  — POS tag indices
      hypothesis_pos   [MAX_LEN]         int64  — POS tag indices
      premise_neg      [MAX_LEN, 2]      float32 — negation flags
      hypothesis_neg   [MAX_LEN, 2]      float32 — negation flags
      premise_elmo     [MAX_LEN, 1024]   float32 — pre-computed ELMo
      hypothesis_elmo  [MAX_LEN, 1024]   float32 — pre-computed ELMo
      premise_mask     [MAX_LEN]         int64   — 1=real, 0=pad
      hypothesis_mask  [MAX_LEN]         int64   — 1=real, 0=pad
      label            []                int64   — 0 or 1
    """
    def __init__(self, df, vocab, char2idx, pos2idx,
                 elmo_prem, elmo_hyp, max_len=MAX_LEN):
        self.df        = df.reset_index(drop=True)
        self.vocab     = vocab
        self.char2idx  = char2idx
        self.pos2idx   = pos2idx
        self.elmo_prem = elmo_prem
        self.elmo_hyp  = elmo_hyp
        self.max_len   = max_len
        self.prem_tokens = [tokenise(s) for s in df['premise']]
        self.hyp_tokens  = [tokenise(s) for s in df['hypothesis']]

    def __len__(self):
        return len(self.df)

    def _get_mask(self, tokens):
        real_len = min(len(tokens), self.max_len)
        return [1] * real_len + [0] * (self.max_len - real_len)

    def _get_token_ids(self, tokens):
        ids = self.vocab.encode(tokens)[:self.max_len]
        return ids + [0] * (self.max_len - len(ids))

    def __getitem__(self, idx):
        prem_tok = self.prem_tokens[idx]
        hyp_tok  = self.hyp_tokens[idx]
        return {
            'premise_ids':      torch.tensor(self._get_token_ids(prem_tok), dtype=torch.long),
            'hypothesis_ids':   torch.tensor(self._get_token_ids(hyp_tok),  dtype=torch.long),
            'premise_char':     words_to_char_tensor(prem_tok, self.char2idx, MAX_WORD_LEN, self.max_len),
            'hypothesis_char':  words_to_char_tensor(hyp_tok,  self.char2idx, MAX_WORD_LEN, self.max_len),
            'premise_pos':      torch.tensor(get_pos_ids(prem_tok, self.pos2idx, self.max_len), dtype=torch.long),
            'hypothesis_pos':   torch.tensor(get_pos_ids(hyp_tok,  self.pos2idx, self.max_len), dtype=torch.long),
            'premise_neg':      get_negation_flags(prem_tok, self.max_len),
            'hypothesis_neg':   get_negation_flags(hyp_tok,  self.max_len),
            'premise_elmo':     torch.tensor(self.elmo_prem[idx], dtype=torch.float32),
            'hypothesis_elmo':  torch.tensor(self.elmo_hyp[idx],  dtype=torch.float32),
            'premise_mask':     torch.tensor(self._get_mask(prem_tok), dtype=torch.long),
            'hypothesis_mask':  torch.tensor(self._get_mask(hyp_tok),  dtype=torch.long),
            'label':            torch.tensor(int(self.df.loc[idx, 'label']), dtype=torch.long),
        }

train_dataset = NLIDataset(train_df, vocab, char2idx, pos2idx, elmo_train_prem, elmo_train_hyp)
dev_dataset   = NLIDataset(dev_df,   vocab, char2idx, pos2idx, elmo_dev_prem,   elmo_dev_hyp)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
dev_loader    = DataLoader(dev_dataset,   batch_size=32, shuffle=False, num_workers=2)
print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Dev:   {len(dev_dataset)} samples, {len(dev_loader)} batches')

In [ ]:
# ── End-to-end sanity check ───────────────────────────────────────────────────
batch = next(iter(train_loader))

input_module.eval()
with torch.no_grad():
    prem_out = input_module(
        batch['premise_ids'], batch['premise_char'],
        batch['premise_pos'], batch['premise_neg'], batch['premise_elmo']
    )
    hyp_out = input_module(
        batch['hypothesis_ids'], batch['hypothesis_char'],
        batch['hypothesis_pos'], batch['hypothesis_neg'], batch['hypothesis_elmo']
    )

print(f'Premise output:    {prem_out.shape}  — expected [32, 64, 1476]')
print(f'Hypothesis output: {hyp_out.shape}  — expected [32, 64, 1476]')
print(f'Premise mask:      {batch["premise_mask"].shape}')
print(f'Labels:            {batch["label"].shape}')
print(f'Sample labels:     {batch["label"][:8]}')

# lengths for OracleNet
prem_lens = batch['premise_mask'].sum(dim=1)
hyp_lens  = batch['hypothesis_mask'].sum(dim=1)
print(f'\nPremise lengths (first 8): {prem_lens[:8]}')
print(f'Hypothesis lengths (first 8): {hyp_lens[:8]}')
print('\nOracleNet call:')
print('  logits = oracle_net(prem_out, hyp_out, prem_lens, hyp_lens)')
print('  input_dim=1476, num_classes=2')

## 11. Save to Drive

Save all artefacts to Google Drive for session persistence.
After a session reset, reload using the commented code in Section 8.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/COMP34812_NLI', exist_ok=True)
DRIVE = '/content/drive/MyDrive/COMP34812_NLI'

# Bundle 1 — vocab and GloVe matrix (small, ~80MB)
torch.save({
    'vocab':        vocab,
    'char2idx':     char2idx,
    'pos2idx':      pos2idx,
    'glove_matrix': glove_matrix,
}, f'{DRIVE}/vocab_and_glove.pt')
print('vocab_and_glove.pt saved')

# Bundle 2 — ELMo arrays compressed (~3.5GB)
np.savez_compressed(f'{DRIVE}/elmo_arrays.npz',
    train_prem = elmo_train_prem,
    train_hyp  = elmo_train_hyp,
    dev_prem   = elmo_dev_prem,
    dev_hyp    = elmo_dev_hyp
)
print('elmo_arrays.npz saved')

for f in ['vocab_and_glove.pt', 'elmo_arrays.npz']:
    size = os.path.getsize(f'{DRIVE}/{f}')
    print(f'  {f}: {size/1e9:.2f} GB')
print('\nAll saved to Drive.')